In [1]:
import segyio
segy_path = "/home/aashrit48/Documents/GitHub/seisvtk/Testing/Teapot3D_Wyoming/seismic/filt_mig.sgy"

from seisvtk.segy_scan import *

geom = detect_geometry(segy_path)
geom

{'iline_byte': 181,
 'xline_byte': 185,
 'n_iline': 345,
 'n_xline': 188,
 'iline_range': (1, 345),
 'xline_range': (1, 188),
 'fill': 1.0,
 'missing': 0,
 'score': 0.81}

In [3]:
from segyio import TraceField as TF

CANDIDATES: dict[int, tuple[int, float]] = {
    189: (TF.INLINE_3D,           1.00),  # SEG-Y rev1 standard for 3D
    193: (TF.CROSSLINE_3D,        1.00),
    181: (TF.CDP_X,               0.90),  # commonly repurposed for IL/XL
    185: (TF.CDP_Y,               0.90),
    9:   (TF.FieldRecord,         0.80),  # classic pre-rev1 convention
    21:  (TF.CDP,                 0.80),
    17:  (TF.EnergySourcePoint,   0.60),
    197: (TF.ShotPoint,           0.60),
    13:  (TF.TraceNumber,         0.30),  # sequence counters
    25:  (TF.CDP_TRACE,           0.30),
    1:   (TF.TRACE_SEQUENCE_LINE, 0.20),
    5:   (TF.TRACE_SEQUENCE_FILE, 0.20),
}


In [3]:
with segyio.open(segy_path,"r",ignore_geometry=True) as f:
    n_traces = f.tracecount
    fields = {}
    for byte, (field, prior) in CANDIDATES.items():
        values = f.attributes(field)[:]
        uniq = np.unique(values)
        # Drop constants and one-value-per-trace counters.
        if 2 <= len(uniq) < n_traces:
            fields[byte] = (values, uniq, prior)

In [9]:
import struct

with open(segy_path,"rb") as fh:
    fh.seek(3788)
    inline = int.from_bytes(fh.read(4),"big", signed=True)
    print(inline)

788937
